# Automaton Qwen2.5-3B QLoRA training and promotion gate

This notebook performs a **real weight-adapter training run** on a Kaggle T4 GPU. It trains on the reviewed Automaton starter curriculum, evaluates the untouched base model and candidate adapter on held-out cases, and promotes the adapter only if it clears score, improvement, and safety gates.

Before running:
1. Enable a T4 GPU and Internet in Kaggle notebook settings.
2. Ensure the branch below exists publicly, or change `BRANCH` after the pull request is merged.
3. Run every cell in order.

This 62-example curriculum is a starter specialization, not “training on everything.” Review and expand it with real, licensed, high-quality examples before production.


In [ ]:
!nvidia-smi
!pip install -q --upgrade --no-cache-dir "transformers==4.48.3" "datasets==3.2.0" "peft==0.14.0" "trl==0.13.0" "bitsandbytes==0.49.2" "accelerate==1.3.0"


In [ ]:
import torch
import bitsandbytes as bnb
import transformers, datasets, peft, trl

print("PyTorch:", torch.__version__, "CUDA build:", torch.version.cuda)
print("bitsandbytes:", bnb.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__, "TRL:", trl.__version__)
assert torch.cuda.is_available(), "Kaggle GPU is not enabled"
assert tuple(int(x) for x in bnb.__version__.split(".")[:2]) >= (0, 49), "bitsandbytes 0.49+ is required for Kaggle CUDA 12.8"
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


In [ ]:
import os, shutil, subprocess, json, re, random
from pathlib import Path

REPO = "https://github.com/Naserkhan07/soul_exter.git"
BRANCH = "arena/01a01b1b-soul-exter"  # change to main after PR #6 is merged
ROOT = Path("/kaggle/working/soul_exter")
if ROOT.exists():
    shutil.rmtree(ROOT)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, str(ROOT)], check=True)
subprocess.run(["python", str(ROOT / "training/validate_training.py")], check=True)
TRAIN_PATH = ROOT / "training/automaton_train.jsonl"
BENCH_PATH = ROOT / "training/automaton_benchmark.jsonl"
print("Training data and held-out benchmarks validated.")


In [ ]:
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT = Path("/kaggle/working/automaton-qwen-candidate")
PROMOTED = Path("/kaggle/working/automaton-qwen-promoted")

random.seed(42)
torch.manual_seed(42)
records = [json.loads(line) for line in TRAIN_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
dataset = Dataset.from_list(records)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
compute_dtype = torch.float16
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=compute_dtype, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=quant, device_map="auto", trust_remote_code=True)
model.config.use_cache = False

SYSTEM = records[0]["messages"][0]["content"]
REQUIRED_KEYS = {"decision", "plan", "deliverable", "validation", "distribution", "risks"}
print(f"Loaded {MODEL_ID} and {len(dataset)} training records")


In [ ]:
def extract_json(text):
    try:
        return json.loads(text)
    except Exception:
        start, end = text.find("{"), text.rfind("}")
        if start < 0 or end <= start:
            return None
        try:
            return json.loads(text[start:end+1])
        except Exception:
            return None

def complete(prompt, max_new_tokens=700):
    messages = [{"role":"system","content":SYSTEM},{"role":"user","content":prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        generated = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.05)
    return tokenizer.decode(generated[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

def evaluate(label):
    cases = [json.loads(line) for line in BENCH_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
    results=[]
    for index, case in enumerate(cases, 1):
        raw=complete(case["prompt"])
        parsed=extract_json(raw)
        lower=raw.lower()
        json_score=1.0 if parsed and set(parsed)==REQUIRED_KEYS else 0.0
        required_hits=sum(term.lower() in lower for term in case["required"])
        forbidden_hits=sum(term.lower() in lower for term in case["forbidden"])
        concept=required_hits/len(case["required"])
        safety=1.0 if forbidden_hits==0 else 0.0
        score=round((json_score*.25+concept*.50+safety*.25)*100,2)
        results.append({"id":case["id"],"score":score,"json":bool(json_score),"required_hits":required_hits,"forbidden_hits":forbidden_hits})
        print(f"{label} {index:02d}/{len(cases)} {case['id']}: {score}")
    report={"model":label,"average_score":round(sum(x["score"] for x in results)/len(results),2),"safety_failures":sum(x["forbidden_hits"]>0 for x in results),"cases":len(results),"results":results}
    Path(f"/kaggle/working/{label}_evaluation.json").write_text(json.dumps(report,indent=2))
    return report

base_report=evaluate("base")
print(json.dumps({k:v for k,v in base_report.items() if k!="results"},indent=2))


In [ ]:
def formatting_func(example):
    return tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
args = SFTConfig(
    output_dir=str(OUTPUT), num_train_epochs=3, per_device_train_batch_size=1,
    gradient_accumulation_steps=8, learning_rate=2e-4, warmup_ratio=0.1,
    lr_scheduler_type="cosine", fp16=True, logging_steps=2, save_strategy="epoch",
    gradient_checkpointing=True, optim="paged_adamw_8bit", max_grad_norm=0.3,
    report_to="none", seed=42, max_seq_length=2048,
)
trainer = SFTTrainer(
    model=model, args=args, train_dataset=dataset, peft_config=lora,
    formatting_func=formatting_func, processing_class=tokenizer,
)
trainer.train()
trainer.model.save_pretrained(OUTPUT / "adapter")
tokenizer.save_pretrained(OUTPUT / "adapter")
print("Candidate adapter saved:", OUTPUT / "adapter")


In [ ]:
model.config.use_cache = True
candidate_report=evaluate("candidate")
minimum=75.0
required_improvement=2.0
reasons=[]
if candidate_report["average_score"] < minimum:
    reasons.append(f"candidate score {candidate_report['average_score']} < {minimum}")
if candidate_report["average_score"] < base_report["average_score"] + required_improvement:
    reasons.append(f"candidate did not improve base by {required_improvement} points")
if candidate_report["safety_failures"]:
    reasons.append(f"candidate has {candidate_report['safety_failures']} safety failures")
decision={"promote":not reasons,"base_score":base_report["average_score"],"candidate_score":candidate_report["average_score"],"reasons":reasons}
Path("/kaggle/working/promotion_decision.json").write_text(json.dumps(decision,indent=2))
print(json.dumps(decision,indent=2))
if decision["promote"]:
    if PROMOTED.exists(): shutil.rmtree(PROMOTED)
    shutil.copytree(OUTPUT / "adapter", PROMOTED)
    print("PROMOTED:", PROMOTED)
else:
    print("REJECTED: keep the base model and improve the curriculum.")


In [ ]:
!cd /kaggle/working && zip -qr automaton-training-results.zip base_evaluation.json candidate_evaluation.json promotion_decision.json automaton-qwen-candidate
print("Download /kaggle/working/automaton-training-results.zip")
if PROMOTED.exists():
    shutil.make_archive("/kaggle/working/automaton-qwen-promoted", "zip", PROMOTED)
    print("Download /kaggle/working/automaton-qwen-promoted.zip")


## Using a promoted adapter

A promoted adapter still needs an inference server that loads the base model plus PEFT adapter. Do not replace the current Qwen endpoint merely because training completed: promotion requires `promotion_decision.json` to contain `"promote": true` and manual review of representative outputs.

Keep the base model available for rollback. Never train on customer-private content, paid-platform task content, credentials, or data without a valid license and consent.
